<a href="https://colab.research.google.com/github/VikaSvyat/DI_Bootcamp/blob/main/Week14/BERT_SentimentAssistant_W14D1_MP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mini Project: Sentiment Assistant with BERT Fine-Tuning

https://octopus.developers.institute/courses/collection/124/course/725/section/1980/chapter/4741

In [1]:
!pip install -q tensorflow tensorflow-datasets accelerate evaluate
!pip install "transformers==4.41.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 38.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import platform
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))

Python version      : 3.12.13
TensorFlow version  : 2.19.0
GPU devices detected: []


In [3]:
import transformers
print(transformers.__version__)
print(transformers.__file__)

4.41.2
/usr/local/lib/python3.12/dist-packages/transformers/__init__.py


In [4]:
!pip show transformers

Name: transformers
Version: 4.41.2
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft, sentence-transformers


Load the IMDB Reviews Dataset

In [5]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)
print(ds_info)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.51W4WG_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.51W4WG_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.51W4WG_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.
tfds.core.DatasetInfo(
    name='imdb_reviews',
    full_name='imdb_reviews/plain_text/1.0.0',
    description="""
    Large Movie Review Dataset. This is a dataset for binary sentiment
    classification containing substantially more data than previous benchmark
    datasets. We provide a set of 25,000 highly polar movie reviews for training,
    and 25,000 for testing. There is additional unlabeled data for use as well.
    """,
    config_description="""
    Plain text
    """,
    homepage='http://ai.stanford.edu/~amaas/data/sentiment/',
    data_dir='/root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0',
    file_format=tfrecord,
    download_size=80.23 MiB,
    dataset_size=129.83 MiB,
    features=FeaturesDict({
        'label': ClassLabel(shape=(), dtype=int64, num_classes=2),
        'text': Text(shape=(), dtype=string),
    }),
   

In [6]:
##samples to make sentiment concrete
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

Label: Negative
This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline ...

Label: Negative
I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell asleep because the film wa ...



Tokenizer Setup & Data Pipeline

In [7]:
####Config

In [8]:
MAX_LENGTH = 256   # trim or pad every review to 256 tokens so batches align
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Tokenizer loaded: bert-base-uncased


convert raw bytes → token IDs, attention masks, and segment IDs.

In [9]:
# def encode_review(review_input):
#     if isinstance(review_input, bytes):
#         review_text = review_input.decode("utf-8")
#     elif hasattr(review_input, "numpy"):
#         review_text = review_input.numpy().decode("utf-8")
#     else:
#         review_text = str(review_input)

#     return tokenizer.encode_plus(
#         review_text,
#         add_special_tokens=True,
#         max_length=MAX_LENGTH,
#         padding="max_length",
#         truncation=True,
#         return_attention_mask=True,
#         return_token_type_ids=True,
#     )

# def tf_encode(text, label):
#     encoded = tf.py_function(
#         func=lambda t: list(encode_review(t).values()),
#         inp=[text],
#         Tout=[tf.int32, tf.int32, tf.int32]
#     )
#     return {
#         "input_ids": encoded[0],
#         "attention_mask": encoded[1],
#         "token_type_ids": encoded[2]
#     }, label

# def prepare_dataset(dataset):
#     return (
#         dataset
#         .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
#         .shuffle(2000)
#         .batch(BATCH_SIZE)
#         .prefetch(tf.data.AUTOTUNE)
#     )

##### Tokenization
def tokenize(text, label):
    text = tf.strings.reduce_join(text)

    text = text.numpy().decode("utf-8")

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    return (
        tokens["input_ids"],
        tokens["attention_mask"],
        tokens["token_type_ids"],
        label
    )
####
def tf_tokenize(text, label):
    input_ids, attention_mask, token_type_ids, label = tf.py_function(
        func=tokenize,
        inp=[text, label],
        Tout=[tf.int32, tf.int32, tf.int32, tf.int64]
    )

    input_ids.set_shape([MAX_LENGTH])
    attention_mask.set_shape([MAX_LENGTH])
    token_type_ids.set_shape([MAX_LENGTH])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": token_type_ids,
    }, label


### Dataset pipeline
def prepare_dataset(ds, shuffle=False):
    ds = ds.map(tf_tokenize, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(2000)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

#### Train / Val / Test split
val_size = int(0.1 * len(ds_train))

ds_val = ds_train.take(val_size)
ds_train = ds_train.skip(val_size)

train_ds = prepare_dataset(ds_train, shuffle=True)
val_ds   = prepare_dataset(ds_val)
test_ds  = prepare_dataset(ds_test)

print("Train dataset size:", len(ds_train))
print("Val   dataset size:", len(ds_val))
print("Test  dataset size:", len(ds_test))

Train dataset size: 22500
Val   dataset size: 2500
Test  dataset size: 25000


Initialize the Fine-Tuning Model

In [10]:
#### Model
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)
#### Compile
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()


tf_model.h5:   0%|          | 0.00/536M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFBertForSequenceClassification.

Some layers of TFBertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: "tf_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  109482240 
                                                                 
 dropout_37 (Dropout)        multiple                  0 (unused)
                                                                 
 classifier (Dense)          multiple                  1538      
                                                                 
Total params: 109483778 (417.65 MB)
Trainable params: 109483778 (417.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
##### Train
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

Epoch 1/3
 315/1407 [=====>........................] - ETA: 19:03:10 - loss: 0.3682 - accuracy: 0.8302

In [ ]:
##### Test evaluation
eval_metrics = model.evaluate(test_ds)

**What lever most improved results?**


The biggest improvement usually comes from data quality and preprocessing, not just model tweaks. Cleaning the text (removing noise, fixing labels, balancing classes) has a larger impact than simply increasing epochs or slightly tuning hyperparameters.

Data cleaning → biggest gain (better signal)


Hyperparameters (learning rate, batch size) → moderate improvement
More epochs → limited benefit, can even cause overfitting

**Where would you add guardrails before deployment?**

Before using this sentiment model in production, guardrails are essential:

- Confidence threshold

Only act on predictions when confidence is high (e.g. > 0.7)

- Human-in-the-loop

Route low-confidence or high-impact cases to human review

- Bias monitoring

Check performance across different user groups or language styles

- Input validation

Reject or flag:

-very short inputs

-non-English text (if model trained on English)

-irrelevant content

- Drift detection

Monitor if real-world data starts differing from training data

**Which stakeholders benefit the most?**
- Support Lead

Identifies unhappy customers quickly
Prioritizes urgent tickets
Improves response time and satisfaction

- Product Manager

Gains insights into user pain points
Tracks sentiment trends over time
Guides feature improvements

- Compliance Officer

Detects risky or sensitive interactions
Flags potential escalation cases
Ensures communication standards